# 🧪 Template oficial de notebook Maia

Copia este notebook, rellena **solo** los bloques marcados con `✏️ TU PARTE` y súbelo a tu repositorio de GitHub.
Todo lo marcado con `🔒 NO TOCAR` es el contrato con la plataforma: si lo cambias, el pipeline falla o el modelo no se despliega.

## Cómo lo ejecuta Maia
1. Descarga el notebook de tu repo (rama elegida) y **valida las etiquetas** de las celdas.
2. Monta los datos que elegiste en la vista del pipeline (nodo *Input*) y te pasa la ruta en `DATASET_PATH`.
3. Ejecuta el notebook con **papermill**, inyectando las variables de la celda `parameters`.
4. Lee de **MLflow** las métricas que registraste y decide si despliega según la *política de despliegue* del proyecto.
5. Sube `model.joblib` al model-server: `POST /predict/<MODEL_NAME>` con `{"data": [[...], ...]}`.

## Etiquetas de celda (`metadata.tags`) — en Jupyter: *View → Cell Toolbar → Tags*
| Etiqueta | ¿Obligatoria? | Qué va ahí |
|---|---|---|
| `parameters` | Sí si usas datos de Maia | Variables que inyecta la plataforma + tus hiperparámetros |
| `mlops:config` | **Sí** | `MODEL_NAME` y `VERSION` |
| `mlops:data` | Opcional | Lectura de `DATASET_PATH` y `TARGET_COLUMN` |
| `mlops:preprocessing` | **Sí** | Columnas de entrada y separación train/test |
| `mlops:training` | **Sí** | El modelo y `fit()` |
| `mlops:evaluation` | Opcional (recomendada) | Métricas → MLflow |
| `mlops:export` | **Sí** | Guardar modelo, features y notas |

Puedes añadir celdas **sin etiqueta** (imports, funciones auxiliares): se ejecutan en orden, pero no aparecen como fase.

## Cómo llegan los datos (datasources)
- Los datasources que agregas (bases de datos, archivos subidos) pasan por la **Data Factory**: bronze → silver → gold.
- En la vista del pipeline arrastras al nodo *Input* lo que quieres usar: **gold** (última versión o una concreta), o el **silver/bronze de un datasource**.
- Sin elección, se usa el **dataset activo** del proyecto.
- Siempre llega como **un único archivo local** en `DATASET_PATH` (`gold-v0003.parquet`, `silver-source-2.parquet`, `bronze-source-2.parquet`, o el archivo subido). Nunca necesitas credenciales de S3/MinIO.
- **Formato**: Maia mira qué lector llama tu código. Si solo usas `pd.read_parquet` te da Parquet; si solo usas `pd.read_csv` te lo convierte a CSV (pierde tipos). Usa **uno solo** — este template usa Parquet.
- Si **no** declaras `DATASET_PATH` en la celda `parameters`, Maia entiende que traes tus propios datos y no monta nada.

## Reglas para que el modelo se pueda desplegar
- El model-server recibe **solo números**, en el **orden de `FEATURE_COLUMNS`**. Codifica las categorías a números antes de `fit()`.
- Todo preprocesamiento que deba repetirse al predecir (imputar, escalar…) va **dentro** del `Pipeline` de sklearn que se guarda.
- La métrica que decide el despliegue se configura en el proyecto (por defecto `accuracy`, más alto es mejor). Debe llamarse **igual** que una de las que registras en `METRICS`.

In [ ]:
# ================================================================
# 🔒 NO TOCAR — celda "parameters" (etiqueta: parameters)
# ================================================================
# Papermill inyecta una celda NUEVA justo después de esta con los valores reales.
# Aquí solo se declaran con un valor vacío para que el notebook corra también a mano.
DATASET_PATH = ""            # Archivo local con los datos elegidos en el nodo Input
MODEL_OUTPUT_PATH = ""       # Dónde guardar el modelo entrenado (.joblib)
PIPELINE_ID = ""             # UUID de la ejecución
MLFLOW_TRACKING_URI = ""     # URL del servidor MLflow
MLFLOW_RUN_ID = ""           # Run de MLflow donde DEBES registrar las métricas
TRAINING_FEATURES_PATH = ""  # Dónde escribir el DataFrame exacto pasado a fit()
MODEL_NOTES_PATH = ""        # Dónde escribir qué significa cada entrada / clase

# ================================================================
# ✏️ TU PARTE — hiperparámetros (opcional)
# ================================================================
# Declara aquí, con un valor por defecto, los parámetros que quieras cambiar desde
# "Run pipeline" sin editar el código. Tienen que estar EN ESTA CELDA: si los defines
# más abajo, tu asignación pisa el valor que inyecta Maia.
# Nombres válidos: identificadores Python, máximo 30, sin usar los nombres de arriba.
# Ejemplo:
# N_ESTIMATORS = 200
# MAX_DEPTH = 8
# TEST_SIZE = 0.2

In [ ]:
# ================================================================
# mlops:config — metadatos del modelo (etiqueta: mlops:config)
# ================================================================
import os

# ✏️ TU PARTE — nombre del modelo. Solo letras, números, "_", "." o "-" (sin espacios).
# Se usa en la URL de predicción: /predict/<MODEL_NAME>. Tras el primer despliegue
# el proyecto conserva su nombre aunque lo cambies aquí.
MODEL_NAME = "mi-modelo"   # ✏️
VERSION = "1"              # ✏️ informativo; Maia numera las versiones sola

# 🔒 NO TOCAR — valores por defecto para ejecutar el notebook fuera de Maia.
MODEL_OUTPUT_PATH = MODEL_OUTPUT_PATH or "./model.joblib"
PIPELINE_ID = PIPELINE_ID or "local-dev"
MLFLOW_TRACKING_URI = MLFLOW_TRACKING_URI or os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")

print(f"Modelo: {MODEL_NAME} v{VERSION} | datos: {DATASET_PATH or '(ninguno)'}")

In [ ]:
# ================================================================
# mlops:data — lectura de los datos (etiqueta: mlops:data)
# ================================================================
from pathlib import Path

import pandas as pd

# ✏️ TU PARTE — la columna que el modelo debe predecir.
TARGET_COLUMN = ""   # ✏️ p. ej. "encounter_class"

# 🔒 NO TOCAR — DATASET_PATH es un archivo local; Maia ya lo descargó de MinIO.
if not DATASET_PATH or not Path(DATASET_PATH).exists():
    raise ValueError(
        "DATASET_PATH está vacío. Arrastra gold o el silver/bronze de un datasource "
        "al nodo Input en la vista del pipeline y vuelve a ejecutar."
    )
LAYER = Path(DATASET_PATH).name.split("-")[0]   # gold | silver | bronze | nombre del archivo subido
df = pd.read_parquet(DATASET_PATH)  # usa un solo lector en todo el notebook

if TARGET_COLUMN not in df.columns:
    raise ValueError(f"TARGET_COLUMN={TARGET_COLUMN!r} no existe. Columnas disponibles: {list(df.columns)}")

print(f"Capa: {LAYER} | filas: {len(df):,} | columnas: {df.shape[1]}")
print(df.dtypes.to_string())

In [ ]:
# ================================================================
# mlops:preprocessing — features y split (etiqueta: mlops:preprocessing)
# ================================================================
from sklearn.model_selection import train_test_split

data = df[df[TARGET_COLUMN].notna()].copy()

# ✏️ TU PARTE — columnas de entrada, en el orden en que se enviarán al predecir.
# Deben ser NUMÉRICAS (el model-server solo acepta números). Deja fuera ids y
# columnas que contengan la respuesta.
FEATURE_COLUMNS = []   # ✏️ p. ej. ["base_cost", "total_claim_cost"]

# ✏️ TU PARTE — transformaciones propias (columnas derivadas, conversión a número…).
# Si creas columnas nuevas, añádelas a FEATURE_COLUMNS.
# data["duracion_min"] = (pd.to_datetime(data["stop"]) - pd.to_datetime(data["start"])).dt.total_seconds() / 60

if not FEATURE_COLUMNS:
    raise ValueError("Rellena FEATURE_COLUMNS con las columnas de entrada del modelo.")

X = data[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
y = data[TARGET_COLUMN]

# ✏️ TU PARTE (opcional) — tamaño del test y semilla.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,} | features: {FEATURE_COLUMNS}")

In [ ]:
# ================================================================
# mlops:training — entrenamiento (etiqueta: mlops:training)
# ================================================================
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# ✏️ TU PARTE — elige tu estimador. Cualquier objeto sklearn con fit/predict sirve.
# Clasificación: RandomForestClassifier, LogisticRegression, GradientBoostingClassifier…
# Regresión:     RandomForestRegressor, LinearRegression…
from sklearn.ensemble import RandomForestClassifier   # ✏️

estimator = RandomForestClassifier(random_state=42)   # ✏️ usa tus hiperparámetros de la celda parameters

# 🔒 NO TOCAR la idea: el preprocesamiento va DENTRO del Pipeline que se guarda,
# porque el model-server llama a model.predict() con los datos crudos.
# ✏️ Puedes añadir pasos (p. ej. ("scaler", StandardScaler())) antes de "model".
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", estimator),
])
model.fit(X_train, y_train)
print("Entrenamiento completado.")

In [ ]:
# ================================================================
# mlops:evaluation — métricas de salida (etiqueta: mlops:evaluation)
# ================================================================
import mlflow

y_pred = model.predict(X_test)

# ✏️ TU PARTE — las métricas que quieres ver en Maia. Nombre → número.
# La política de despliegue del proyecto usa UNA de ellas (por defecto "accuracy",
# más alto = mejor). Para regresión, registra p. ej. "rmse" y configura la
# política con métrica "rmse" y dirección "lower".
from sklearn.metrics import accuracy_score, f1_score   # ✏️

METRICS = {   # ✏️
    "accuracy": accuracy_score(y_test, y_pred),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted"),
    # "precision": ..., "recall": ..., "rmse": ..., "r2": ...
}

# ✏️ TU PARTE (opcional) — parámetros informativos que quieras guardar en MLflow.
PARAMS = {"layer": LAYER, "target_column": TARGET_COLUMN, "features": ",".join(FEATURE_COLUMNS)}

# 🔒 NO TOCAR — registra en el run que abrió Maia (MLFLOW_RUN_ID); Maia lee las
# métricas de ahí. Sin MLFLOW_RUN_ID (ejecución manual) abre un run propio.
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
started_here = mlflow.active_run() is None
if started_here:
    if MLFLOW_RUN_ID:
        mlflow.start_run(run_id=MLFLOW_RUN_ID)
    else:
        mlflow.start_run(run_name=f"{MODEL_NAME}-{PIPELINE_ID}")
mlflow.log_params(PARAMS)
mlflow.log_metrics({name: float(value) for name, value in METRICS.items()})
mlflow.log_metric("n_samples", float(len(X)))
if started_here:
    mlflow.end_run()

print({name: round(float(value), 4) for name, value in METRICS.items()})

In [ ]:
# ================================================================
# mlops:export — artefactos (etiqueta: mlops:export)
# ================================================================
import json

import joblib

# 🔒 NO TOCAR — el modelo que se registra en MLflow y se despliega.
joblib.dump(model, MODEL_OUTPUT_PATH)
print(f"Modelo guardado en {MODEL_OUTPUT_PATH}")

# 🔒 NO TOCAR — el DataFrame exacto pasado a fit() (mismas columnas, mismo orden).
# Con él Maia construye la página de prueba del modelo (rangos y efecto de cada entrada).
if TRAINING_FEATURES_PATH:
    X_train.to_parquet(TRAINING_FEATURES_PATH)

# ✏️ TU PARTE (opcional) — qué significa cada cosa, para quien use el modelo.
# Las claves de "inputs" deben coincidir con FEATURE_COLUMNS y las de "classes"
# con las etiquetas que predice el modelo. Lo que no coincida se ignora.
NOTES = {   # ✏️
    "target": {"name": TARGET_COLUMN, "description": ""},
    "inputs": {
        # "base_cost": {"description": "Costo del encuentro antes de seguros", "unit": "USD"},
    },
    "classes": {
        # "inpatient": "Hospitalizado al menos una noche",
    },
}

if MODEL_NOTES_PATH:
    with open(MODEL_NOTES_PATH, "w", encoding="utf-8") as handle:
        json.dump(NOTES, handle, ensure_ascii=False)